In [64]:
%pip install nltk
%pip install websockets
%pip install playwright
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [65]:
import os
import json
import asyncio
import websockets
import nltk
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from playwright.async_api import async_playwright
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline

In [66]:
# Resource setup
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Download VADER lexicon
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

nltk.download(['punkt', 'stopwords', 'punkt_tab', 'wordnet', 'omw-1.4'])
lemmatizer = WordNetLemmatizer()


sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 # Set to 0 if you have a GPU
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [67]:
BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
AUTH_FILE = "auth.json"
MAX_CONCURRENT_TABS = 3
semaphore = asyncio.Semaphore(MAX_CONCURRENT_TABS)
word_counts = Counter()

# Refined ignore list
ignore_words = [
    "i", "me", "my", "myself", "we", "us", "our", "ours", "ourselves", "you", "your", "yours", 
    "yourself", "yourselves", "he", "him", "his", "she", "her", "hers", "it", "its", "they", 
    "them", "their", "someone", "anyone", "am", "is", "are", "was", "were", "be", "been", 
    "being", "have", "has", "had", "do", "does", "did", "shall", "will", "should", "would", 
    "may", "might", "must", "can", "could", "get", "got", "getting", "go", "goes", "went", 
    "gone", "take", "took", "taken", "make", "made", "making", "like", "know", "knows", 
    "knew", "want", "wants", "wanted", "look", "looks", "really", "very", "just", "actually", 
    "basically", "literally", "even", "also", "still", "maybe", "probably", "quite", "much", 
    "many", "lot", "lots", "thing", "things", "stuff", "reddit", "subreddit", "sub", "post", 
    "posts", "comment", "comments", "thread", "edit", "deleted", "removed", "amp", "https", 
    "http", "com", "www"
]

In [68]:
def get_sentiment(text):
    """
    Scans entire text and understands context using a 
    Transformer model. Returns Positive, Neutral, or Negative.
    """
    # Truncate text to 512 tokens (Model limit)
    truncated_text = text[:512]
    result = sentiment_pipeline(truncated_text)[0]
    
    # Mapping the model labels to your format
    label_map = {
        "positive": "Positive",
        "neutral": "Neutral",
        "negative": "Negative"
    }
    return label_map.get(result['label'].lower(), "Neutral")

In [69]:
def get_cache_path(subreddit):
    # Keep cache local to the Python project
    if not os.path.exists("cache"): os.makedirs("cache")
    return os.path.join("cache", f"{subreddit}.json")

def load_cache(subreddit):
    path = get_cache_path(subreddit)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f: return json.load(f)
    return {}

def save_cache(subreddit, data):
    with open(get_cache_path(subreddit), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [70]:
async def get_post_metadata(post):
    pid = await post.get_attribute("id")
    time_loc = post.locator("time").first
    ts = ""
    if await time_loc.count() > 0:
        ts = await time_loc.get_attribute("datetime")
    return pid, ts

async def scrape_single_post(context, url, pid, ts, websocket, total_priority, tracker, cache, subreddit, is_priority):
    async with semaphore:
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            
            title = "[Title Not Found]"
            for sel in ['h1[slot="title"]', 'h1[id^="post-title-"]', 'shreddit-title', 'h1']:
                loc = page.locator(sel).first
                if await loc.is_visible(timeout=3000):
                    title = await loc.inner_text(); break

            content = ""
            for sel in ['shreddit-post-text-body', 'div[slot="text-body"]']:
                loc = page.locator(sel).first
                if await loc.count() > 0:
                    content = await loc.inner_text(timeout=3000)
                    if content: break
            
            # NLP & Sentiment
            full_text = title + " " + content
            sentiment = get_sentiment(full_text)
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [lemmatizer.lemmatize(w) for w in words if w.isalnum() and w not in sw and w not in ignore_words]
            keywords = dict(Counter(clean))
            
            # --- PERSIST TO CACHE IMMEDIATELY ---
            post_entry = {
                "id": pid, 
                "timestamp": ts, 
                "title": title, 
                "body": content, 
                "keywords": keywords, 
                "sentiment": sentiment
            }
            cache[pid] = post_entry
            save_cache(subreddit, cache)

            # --- SEND DELTAS ONLY FOR PRIORITY NEW POSTS ---
            if is_priority:
                tracker['ui_processed'] += 1
                progress = int((tracker['ui_processed'] / total_priority) * 100) if total_priority > 0 else 100
                
                if websocket.state.name == 'OPEN':
                    # Send the full post object directly
                    await websocket.send(json.dumps({
                        "type": "delta_update",
                        "post": post_entry,
                        "progress": progress
                    }))
                    print(f"New Post Scraped: {title[:30]} | Progress: {progress}%")
            else:
                print(f"Background Cached (Gap Fill): {title[:30]}")
            
            return post_entry
        except Exception as e:
            print(f"Error {url}: {e}"); return None
        finally: await page.close()

In [71]:
async def run_scraper(websocket, subreddit, count):
    cache = load_cache(subreddit)
    
    # 1. IMMEDIATE STATUS TO FRONTEND
    if websocket.state.name == 'OPEN':
        await websocket.send(json.dumps({
            "type": "status", 
            "message": f"Fetching latest {count} posts from r/{subreddit}..."
        }))

    # Determine the newest post we currently have in cache
    sorted_cache_list = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)
    last_known_id = sorted_cache_list[0]['id'] if sorted_cache_list else None
    
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False, executable_path=BRAVE_PATH)
    ctx_args = {"storage_state": AUTH_FILE} if os.path.exists(AUTH_FILE) else {}
    context = await browser.new_context(**ctx_args)
    
    try:
        main_page = await context.new_page()
        await main_page.goto(f"https://www.reddit.com/r/{subreddit}/new", wait_until="domcontentloaded")

        # --- PHASE 1: TRUE STREAMING (SCRAPE THE SECOND WE FIND IT) ---
        priority_tasks = []
        launched_ids = set()
        tracker = {'ui_processed': 0}
        found_last_known = False

        while len(priority_tasks) < count:
            posts = await main_page.locator('shreddit-post').all()
            new_elements_found = False

            for p in posts:
                if len(priority_tasks) >= count:
                    break

                pid, ts = await get_post_metadata(p)
                if not pid or pid in launched_ids: continue

                launched_ids.add(pid)
                new_elements_found = True

                if pid == last_known_id:
                    found_last_known = True

                if pid not in cache:
                    ptype = await p.get_attribute('post-type')
                    if ptype in ['text', 'image', 'gallery', 'link']:
                        href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                        url = f"https://www.reddit.com{href}"
                        
                        # LAUNCH SCRAPE IMMEDIATELY (Do not wait for loop to finish)
                        task = asyncio.create_task(
                            scrape_single_post(context, url, pid, ts, websocket, count, tracker, cache, subreddit, True)
                        )
                        priority_tasks.append(task)
                        print(f"Discovered & launched priority worker for: {pid}")

            if len(priority_tasks) >= count or found_last_known:
                break

            # Only scroll if we ran out of visible posts
            if not new_elements_found:
                await main_page.mouse.wheel(0, 4000)
                await asyncio.sleep(1.5)

        # Wait for all the spawned UI workers to finish scraping
        if priority_tasks:
            await asyncio.gather(*priority_tasks)

        # --- PHASE 2: SEND FINAL DATA TO UI ---
        if websocket.state.name == 'OPEN':
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            await websocket.send(json.dumps({"type": "progress", "value": 100}))
            await websocket.send(json.dumps({
                "type": "final_data", 
                "posts": final_selection
            }))
            await websocket.send(json.dumps({"type": "status", "message": "Dashboard updated. Background caching started."}))
            print("UI Target met. Final data sent. Moving to background maintenance.")

        # --- PHASE 3: BACKGROUND GAP FILLING (SILENT) ---
        if last_known_id and not found_last_known:
            print("Checking for historical gap between new posts and last cache...")
            background_tasks = []
            found_gap_end = False

            while not found_gap_end and len(background_tasks) < 300: 
                posts = await main_page.locator('shreddit-post').all()

                for p in posts:
                    pid, ts = await get_post_metadata(p)
                    if not pid or pid in launched_ids: continue

                    launched_ids.add(pid)

                    if pid == last_known_id:
                        found_gap_end = True
                        break

                    if pid not in cache:
                        ptype = await p.get_attribute('post-type')
                        if ptype in ['text', 'image', 'gallery', 'link']:
                            href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                            url = f"https://www.reddit.com{href}"
                            
                            # Launch background workers immediately
                            task = asyncio.create_task(
                                scrape_single_post(context, url, pid, ts, websocket, 0, tracker, cache, subreddit, False)
                            )
                            background_tasks.append(task)

                if found_gap_end: break
                await main_page.mouse.wheel(0, 4000)
                await asyncio.sleep(1.5)

            if background_tasks:
                print(f"Caching {len(background_tasks)} historical posts silently in background...")
                await asyncio.gather(*background_tasks)
            else:
                print("No historical gap found.")

    except Exception as e:
        print(f"Playwright Execution Error: {e}")
    finally:
        await browser.close()
        await playwright.stop()
        print("All scraping and maintenance complete.")

In [ ]:
async def handler(websocket):
    print("Waiting for command...")
    async for message in websocket:
        data = json.loads(message)
        if data.get('type') == 'start_scrape':
            sub = data.get('subreddit', 'mumbai')
            limit = data.get('count', 10)
            print(f"Command received: Scraping r/{sub} (Limit: {limit})")
            word_counts.clear()
            await run_scraper(websocket, sub, limit)

async def start_server():
    print("WebSocket Server started on ws://192.168.0.246:8765")
    async with websockets.serve(handler, "192.168.0.246", 8765):
        await asyncio.Future()

await start_server()

WebSocket Server started on ws://192.168.0.246:8765
Waiting for command...
Waiting for command...
Command received: Scraping r/AskIndianWomen (Limit: 100)
Discovered & launched priority worker for: t3_1t1423j
Discovered & launched priority worker for: t3_1t1412c
New Post Scraped: Am I overthinking this person' | Progress: 1%
New Post Scraped: feeling like I'm becoming perf | Progress: 2%
UI Target met. Final data sent. Moving to background maintenance.
All scraping and maintenance complete.


connection handler failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 376, in conn_handler
    await self.handler(connection)
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_29832\435204060.py", line 3, in handler
    async for message in websocket:
    ...<6 lines>...
            await run_scraper(websocket, sub, limit)
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 244, in __aiter__
    yield await self.recv()
          ^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 324, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
